# 임베딩 모델 비교 + ChromaDB/FAISS 인덱싱 + QA Chain

In [1]:
!wget https://raw.githubusercontent.com/ws-l/hh_2026_5/main/data/reports.json

--2026-06-23 06:54:58--  https://raw.githubusercontent.com/ws-l/hh_2026_5/main/data/reports.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 126861 (124K) [text/plain]
Saving to: ‘reports.json.1’

reports.json.1      100%[===================>] 123.89K  --.-KB/s    in 0.05s   

2026-06-23 06:54:58 (2.40 MB/s) - ‘reports.json.1’ saved [126861/126861]



In [2]:
!pip install -q sentence-transformers chromadb faiss-cpu langchain langchain-community langchain-huggingface

In [3]:
!pip install -q langchain-ollama

In [4]:
import json
import time
import numpy as np
from sentence_transformers import SentenceTransformer
import matplotlib.pyplot as plt
import seaborn as sns

with open("reports.json", "r", encoding="utf-8") as f:
    reports = json.load(f)

print(f"총 {len(reports)}건 로드")


총 150건 로드


## LangChain RetrievalQA Chain 구성

ChromaDB를 LangChain의 vector store로 연결하고, QA Chain을 통해 엔드투엔드로 질의응답

ko-sroberta: 한국어 문장 간 의미 유사도에 특화되어 있어서, 검색 품질이 좋음

In [5]:
from langchain_community.vectorstores import Chroma as LC_Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_classic.chains import RetrievalQA  
from langchain_huggingface import HuggingFacePipeline
from transformers import pipeline

# LangChain용 임베딩 래퍼, langchain 전용 규격
lc_embeddings = HuggingFaceEmbeddings(model_name="jhgan/ko-sroberta-multitask")

# Document 객체로 변환
lc_documents = [
    Document(page_content=r["report_text"], metadata={"report_id": r["report_id"]})
    for r in reports
]

# LangChain 전용 별도 컬렉션으로 vector store 구성
lc_vectorstore = LC_Chroma.from_documents(
    documents=lc_documents,
    embedding=lc_embeddings,
    collection_name="lc_engine_reports",
)

retriever = lc_vectorstore.as_retriever(search_kwargs={"k": 3})
print("LangChain vector store 구성 완료")


/tmp/ipykernel_7458/1698427841.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma as LC_Chroma
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:134: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

LangChain vector store 구성 완료


- ollama 설치

In [6]:
!apt-get -qq update
!apt-get -qq install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [7]:
import subprocess
import time

# 기존에 떠있는 프로세스가 있다면 정리 후 재시작
!pkill ollama
time.sleep(2)

process = subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(5)
print("Ollama 서버 시작 (PID:", process.pid, ")")


Ollama 서버 시작 (PID: 8720 )


In [8]:
!ollama pull gemma:2b

In [9]:
from langchain_ollama import OllamaLLM
llm = OllamaLLM(model="gemma:2b", temperature=0.3, system="You are a manufacturing expert. Always respond in Korean.")

In [18]:
from langchain_core.prompts import PromptTemplate

# 한국어 응답 강제 프롬프트
korean_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""당신은 제조 품질 전문가입니다. 아래 문서를 참고하여 질문에 반드시 한국어로 답변하세요.

[참고 문서]
{context}

[질문]
{question}

[한국어 답변]"""
)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": korean_prompt}, 
)
print("QA Chain 구성 완료")

QA Chain 구성 완료


In [16]:
question = "불량 사례의 원인은 보통 무엇인가요?"
response = qa_chain.invoke({"query": question})

print("=== 답변 ===")
print(response["result"])
print()
print("=== 참고한 출처 문서 ===")
for doc in response["source_documents"]:
    print("-", doc.metadata["report_id"], ":", doc.page_content[:150])

=== 답변 ===
불량 사례의 원인은 2024년 12월 8일 10:25, 사출성형기 #3호기에서 PC 투명 커버 게이트 주변에 길이 6~12mm의 크랙이 반복 발생한 것으로 확인되었습니다. 금형 온도가 설정 62°C 대비 47°C 수준으로 낮았고 초기 생산품 150개 중 41개가 외관 불량으로 판정되어 설비보전팀에서 심각도 상으로 관리되었습니다.

=== 참고한 출처 문서 ===
- RPT-128 : 2024년 12월 8일 10:25, 사출성형기 #3호기에서 PC 투명 커버 게이트 주변에 길이 6~12mm의 크랙이 반복 발생함을 확인함. 금형온도가 설정 62°C 대비 47°C 수준으로 낮았고 초기 생산품 150개 중 41개가 외관 불량으로 판정되어 설비보전팀에서 심
- RPT-068 : 2024년 7월 23일 11:20, 기계식 프레스 #8호기에서 커버 외곽 폭이 기준 76.00mm 대비 75.71~75.78mm로 작게 타발된 것을 확인함. 금형 교환 후 첫 제품 검사가 생략되었고 금형 클리어런스는 0.07mm로 낮게 설정되어 품질관리팀에서 심각도 중
- RPT-033 : 2024년 4월 7일 10:10, 컨베이어 벨트 3라인에서 플라스틱 커버 이송 후 외관면에 길이 15~28mm의 선형 스크래치가 발생한 것을 확인함. 이송속도는 2.4m/s였으며 공급 자재 박스 내부 완충지가 누락된 묶음에서 결함이 집중되어 품질관리팀에서 심각도 하로 
